# GNN-Based Credit Card Fraud Detection
**Roya Amiri & Sardar Jaf — University of Sunderland, 2026**

Paper: *Isolating Graph Topology from Model Architecture in GNN-Based Fraud Detection*

---
### Before you start
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier)
2. Run cells top to bottom in order
3. All outputs (plots, CSVs) are saved to `/content/outcomes/`

---
### Contents
- **Cell 1** — Install dependencies  
- **Cell 2** — Mount Google Drive (optional, for IBM parquet)  
- **Cell 3** — Upload and unzip the project  
- **Cell 4** — Download Sparkov dataset from Kaggle  
- **Cell 5** — Run Sparkov EDA  
- **Cell 6** — Run Sparkov GATv2 (all 3 strategies)  
- **Cell 7** — Run Sparkov GCN  
- **Cell 8** — Run Sparkov GraphSAGE  
- **Cell 9** — Run Sparkov MLP baseline  
- **Cell 10** — Run IBM EDA  
- **Cell 11** — Run IBM GATv2  
- **Cell 12** — Run IBM GCN  
- **Cell 13** — Run IBM GraphSAGE  
- **Cell 14** — Run IBM MLP baseline  
- **Cell 15** — View and download all saved outputs

---
## Cell 1 — Install dependencies
Run this first. Takes ~3 minutes on a fresh Colab instance.

In [ ]:
import subprocess, sys

# PyTorch Geometric — must match the installed torch version
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch_version}  |  CUDA: {cuda_version}')

!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    torch-geometric -f https://data.pyg.org/whl/torch-{torch_version}+{cuda_version}.html

!pip install -q faiss-cpu pyarrow seaborn kaggle

print('\nAll dependencies installed.')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## Cell 2 — Mount Google Drive (optional)
Only needed if your IBM dataset is stored in Google Drive.
Skip this cell if you are only running Sparkov.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# After mounting, your files are at /content/drive/MyDrive/
# Example:
# IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'
IBM_DATA_PATH = '/content/drive/MyDrive/reduced_dataset.parquet'  # ← update if needed
print(f'IBM data path set to: {IBM_DATA_PATH}')

---
## Cell 3 — Upload and unzip the project
Upload `fraud_gnn_project.zip` using the Files panel on the left,
then run this cell to unzip it.

In [ ]:
import os

# Option A: you uploaded the zip manually via the Files panel
!unzip -q /content/fraud_gnn_project.zip -d /content/

# Option B: load from Google Drive instead
# !unzip -q /content/drive/MyDrive/fraud_gnn_project.zip -d /content/

# Create outcomes folder
os.makedirs('/content/outcomes', exist_ok=True)
os.makedirs('/content/outcomes/eda_sparkov', exist_ok=True)
os.makedirs('/content/outcomes/eda_ibm', exist_ok=True)

# Verify
print('Project files:')
for root, dirs, files in os.walk('/content/fraud_gnn_project'):
    level = root.replace('/content/fraud_gnn_project', '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    print(f'{indent}{folder}/')
    for f in sorted(files):
        print(f'{indent}  {f}')

---
## Cell 4 — Download Sparkov dataset from Kaggle

**Option A (recommended): Kaggle API**  
1. Go to kaggle.com → Account → Create New Token → downloads `kaggle.json`  
2. Upload `kaggle.json` via the Files panel  
3. Run the cell below

**Option B: Manual upload**  
Download `fraudTrain.csv` and `fraudTest.csv` from  
https://www.kaggle.com/datasets/kartik2112/fraud-detection  
and upload them directly to `/content/fraud_gnn_project/sparkov/`

In [ ]:
import os

# ── Option A: Kaggle API ──────────────────────────────────────────────────────
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

os.chdir('/content/fraud_gnn_project/sparkov')
!kaggle datasets download -d kartik2112/fraud-detection --unzip

# Confirm files are present
for f in ['fraudTrain.csv', 'fraudTest.csv']:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    status = f'{size:.1f} MB' if size > 0 else 'NOT FOUND'
    print(f'  {f}: {status}')

# ── Option B: Manual upload check ────────────────────────────────────────────
# If you uploaded manually, just run this to verify:
# os.chdir('/content/fraud_gnn_project/sparkov')
# for f in ['fraudTrain.csv', 'fraudTest.csv']:
#     size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
#     print(f'{f}: {size:.1f} MB' if size > 0 else f'{f}: NOT FOUND')

---
## Cell 5 — Sparkov EDA
Generates 14 figures saved to `/content/outcomes/eda_sparkov/`

In [ ]:
import os, sys

os.chdir('/content/fraud_gnn_project/sparkov')
sys.path.insert(0, os.getcwd())

# Point outcomes to /content/outcomes
import importlib

# Patch OUTCOME_DIR before running
import eda as sparkov_eda
sparkov_eda.OUT_DIR = '/content/outcomes/eda_sparkov'
os.makedirs(sparkov_eda.OUT_DIR, exist_ok=True)

# Load data
df_sparkov = sparkov_eda.load_data()
sparkov_eda.print_summary(df_sparkov)

# Generate all 14 figures
print('\nGenerating Sparkov EDA figures...')
sparkov_eda.plot_class_distribution(df_sparkov)
sparkov_eda.plot_fraud_by_hour(df_sparkov)
sparkov_eda.plot_fraud_by_weekday(df_sparkov)
sparkov_eda.plot_fraud_by_month(df_sparkov)
sparkov_eda.plot_amount_distribution(df_sparkov)
sparkov_eda.plot_amount_boxplot(df_sparkov)
sparkov_eda.plot_category_fraud(df_sparkov)
sparkov_eda.plot_state_fraud(df_sparkov)
sparkov_eda.plot_age_distribution(df_sparkov)
sparkov_eda.plot_time_diff(df_sparkov)
sparkov_eda.plot_distance(df_sparkov)
sparkov_eda.plot_amt_to_avg(df_sparkov)
sparkov_eda.plot_cardholder_txn_count(df_sparkov)
sparkov_eda.plot_cardholder_fraud_rate(df_sparkov)

# Display figures inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

figs = sorted(os.listdir(sparkov_eda.OUT_DIR))
print(f'\n{len(figs)} figures saved. Displaying...')
for fname in figs:
    if fname.endswith('.png'):
        print(f'\n--- {fname} ---')
        display(Image(filename=os.path.join(sparkov_eda.OUT_DIR, fname)))

---
## Cell 6 — Sparkov GATv2
Runs all 3 graph strategies (multi_relation, hybrid, intra_group) under GATv2.
Expected time: ~25–40 min on T4 GPU.

In [ ]:
import os, sys, importlib

os.chdir('/content/fraud_gnn_project/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg = sparkov_cfg.CardFraudConfig()
cfg.OUTCOME_DIR = '/content/outcomes'

import utils as sparkov_utils
import gatv2_model

# Load data (reuse if already loaded)
try:
    df_sparkov
except NameError:
    df_sparkov = sparkov_utils.load_and_preprocess(
        train_path='fraudTrain.csv', test_path='fraudTest.csv', cfg=cfg)

# Run all 3 strategies
gatv2_results = gatv2_model.run_all_strategies(df_sparkov, cfg)
print('\nGATv2 done. Results saved to /content/outcomes/')

---
## Cell 7 — Sparkov GCN

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg = sparkov_cfg.CardFraudConfig()
cfg.OUTCOME_DIR = '/content/outcomes'

import utils as sparkov_utils
import gcn_model

try:
    df_sparkov
except NameError:
    df_sparkov = sparkov_utils.load_and_preprocess(
        train_path='fraudTrain.csv', test_path='fraudTest.csv', cfg=cfg)

gcn_results = gcn_model.run_all_strategies(df_sparkov, cfg)
print('\nGCN done.')

---
## Cell 8 — Sparkov GraphSAGE

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg = sparkov_cfg.CardFraudConfig()
cfg.OUTCOME_DIR = '/content/outcomes'

import utils as sparkov_utils
import graphsage_model

try:
    df_sparkov
except NameError:
    df_sparkov = sparkov_utils.load_and_preprocess(
        train_path='fraudTrain.csv', test_path='fraudTest.csv', cfg=cfg)

sage_results = graphsage_model.run_all_strategies(df_sparkov, cfg)
print('\nGraphSAGE done.')

---
## Cell 9 — Sparkov MLP Baseline

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
cfg = sparkov_cfg.CardFraudConfig()
cfg.OUTCOME_DIR = '/content/outcomes'

import utils as sparkov_utils
import mlp_baseline as sparkov_mlp

try:
    df_sparkov
except NameError:
    df_sparkov = sparkov_utils.load_and_preprocess(
        train_path='fraudTrain.csv', test_path='fraudTest.csv', cfg=cfg)

mlp_result = sparkov_mlp.run_pipeline(df_sparkov, cfg)
print('\nMLP baseline done.')

---
## Cell 10 — IBM EDA
Make sure Cell 2 (Drive mount) has been run and `IBM_DATA_PATH` is set correctly.

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/ibm')
sys.path.insert(0, os.getcwd())

import eda as ibm_eda

# Set your IBM data path here
ibm_eda.DATA_PATH = IBM_DATA_PATH  # set in Cell 2, or change manually here
ibm_eda.OUT_DIR   = '/content/outcomes/eda_ibm'
os.makedirs(ibm_eda.OUT_DIR, exist_ok=True)

df_ibm = ibm_eda.load_data()
ibm_eda.print_summary(df_ibm)

print('\nGenerating IBM EDA figures...')
ibm_eda.plot_class_distribution(df_ibm)
ibm_eda.plot_fraud_by_hour(df_ibm)
ibm_eda.plot_fraud_by_weekday(df_ibm)
ibm_eda.plot_fraud_by_month(df_ibm)
ibm_eda.plot_amount_distribution(df_ibm)
ibm_eda.plot_amount_boxplot(df_ibm)
ibm_eda.plot_city_fraud(df_ibm)
ibm_eda.plot_chip_fraud(df_ibm)
ibm_eda.plot_error_fraud(df_ibm)
ibm_eda.plot_mcc_fraud(df_ibm)
ibm_eda.plot_user_txn_count(df_ibm)
ibm_eda.plot_card_txn_count(df_ibm)
ibm_eda.plot_user_fraud_rate(df_ibm)
ibm_eda.plot_amt_to_user_avg(df_ibm)

from IPython.display import display, Image
for fname in sorted(os.listdir(ibm_eda.OUT_DIR)):
    if fname.endswith('.png'):
        print(f'\n--- {fname} ---')
        display(Image(filename=os.path.join(ibm_eda.OUT_DIR, fname)))

---
## Cell 11 — IBM GATv2

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes'

import utils as ibm_utils
import gatv2_model as ibm_gatv2

try:
    df_ibm
except NameError:
    df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)

ibm_gatv2_results = ibm_gatv2.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM GATv2 done.')

---
## Cell 12 — IBM GCN

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes'

import utils as ibm_utils
import gcn_model as ibm_gcn

try:
    df_ibm
except NameError:
    df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)

ibm_gcn_results = ibm_gcn.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM GCN done.')

---
## Cell 13 — IBM GraphSAGE

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes'

import utils as ibm_utils
import graphsage_model as ibm_sage

try:
    df_ibm
except NameError:
    df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)

ibm_sage_results = ibm_sage.run_all_strategies(df_ibm, cfg_ibm)
print('\nIBM GraphSAGE done.')

---
## Cell 14 — IBM MLP Baseline

In [ ]:
import os, sys
os.chdir('/content/fraud_gnn_project/ibm')
sys.path.insert(0, os.getcwd())

import config as ibm_cfg
cfg_ibm = ibm_cfg.IBMFraudConfig()
cfg_ibm.OUTCOME_DIR = '/content/outcomes'

import utils as ibm_utils
import mlp_baseline as ibm_mlp

try:
    df_ibm
except NameError:
    df_ibm = ibm_utils.load_and_preprocess(path=IBM_DATA_PATH, cfg=cfg_ibm)

ibm_mlp_result = ibm_mlp.run_pipeline(df_ibm, cfg_ibm)
print('\nIBM MLP baseline done.')

---
## Cell 15 — View all outputs and download
Lists everything in `/content/outcomes/` and zips it for download.

In [ ]:
import os
from IPython.display import display, Image

OUT = '/content/outcomes'

# ── List all saved files ──────────────────────────────────────────────────────
print('All saved outputs:\n')
total_files = 0
for root, dirs, files in os.walk(OUT):
    dirs.sort(); files.sort()
    level  = root.replace(OUT, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        size = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{indent}  {f}  ({size:.0f} KB)")
        total_files += 1
print(f'\nTotal files: {total_files}')

# ── Display CSV summaries ─────────────────────────────────────────────────────
import pandas as pd
for root, dirs, files in os.walk(OUT):
    for f in files:
        if f.endswith('.csv'):
            path = os.path.join(root, f)
            print(f'\n=== {f} ===')
            display(pd.read_csv(path).to_string(index=False))

# ── Zip everything for download ───────────────────────────────────────────────
!cd /content && zip -r outcomes.zip outcomes/ -x "*.DS_Store"

from google.colab import files
print('\nDownloading outcomes.zip...')
files.download('/content/outcomes.zip')

---
## Quick reference: run a single strategy / architecture

If you only want to test one combination without running the full matrix:

In [ ]:
# Example: run only multi_relation + GATv2 on Sparkov
import os, sys
os.chdir('/content/fraud_gnn_project/sparkov')
sys.path.insert(0, os.getcwd())

import config as sparkov_cfg
import utils  as sparkov_utils
import gatv2_model

cfg = sparkov_cfg.CardFraudConfig()
cfg.OUTCOME_DIR = '/content/outcomes'

df = sparkov_utils.load_and_preprocess(
    train_path='fraudTrain.csv', test_path='fraudTest.csv', cfg=cfg)

# Change graph_strategy to: 'multi_relation' | 'hybrid' | 'intra_group'
result = gatv2_model.run_pipeline(df, cfg, graph_strategy='multi_relation')

print('\nTest metrics:')
for k, v in result['test_metrics'].items():
    if k != 'cm':
        print(f'  {k}: {v:.4f}')
print(f"  confusion matrix:\n{result['test_metrics']['cm']}")

---
## Tips for Colab

| Situation | Solution |
|---|---|
| Runtime disconnects mid-run | Save results to Drive: add `cfg.OUTCOME_DIR = '/content/drive/MyDrive/outcomes'` |
| Out of GPU memory | Reduce `cfg.N_SPLITS = 3` or `cfg.DOWNSAMPLE_NONFRAUD = 500_000` |
| Module not found error | Re-run Cell 3 (unzip) and make sure `sys.path` points to the correct folder |
| Slow graph building | Normal — intra_group and hybrid take longer than multi_relation |
| Want faster test run | Set `cfg.MAX_EPOCHS = 30` and `cfg.N_SPLITS = 2` at the top of any model cell |